In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 23.3 MB/s eta 0:00:00


In [ ]:
pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0


In [ ]:
import anthropic
import time
import csv
from tqdm import tqdm
import argparse
from openai import OpenAI

## **CONFIGURATION**



In [ ]:
OPENAI_API_KEY    = "OPENAI_API_KEY"
ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"


MODELS = {
    "gpt-5.4":           "openai",
    "claude-opus-4-6":   "anthropic"
}

# Reflection settings
MAX_REFLECT  = 2     # fixed number of reflection rounds
DELTA        = 0.10  # minimum score change to continue reflecting

openai_client    = OpenAI(
    api_key = OPENAI_API_KEY
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


In [ ]:
def call_llm(prompt, model):

    provider = MODELS.get(model)

    if provider is None:
      print(f"  Unknown model: {model}")
      return None

    if provider == "openai":
      response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
      return response.choices[0].message.content

    elif provider == "anthropic":
      response = anthropic_client.messages.create(
            model=model,
            temperature=0,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
      )
      return response.content[0].text


    return None

## **AUTO-COT STEP GENERATION**

In [ ]:
def generate_cot_steps_no_gold(model):
    """
    Generate CoT evaluation steps WITHOUT gold reference.
    Used by S2 and S3 (same steps, S3 reuses S2's).
    Run once — saved to file for inspection and reuse.
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate the CLASSES in a PlantUML candidate diagram given only
a problem description.

The evaluator will apply this correctness rule:
- Supported by problem description correct → CORRECT
- Not supported by problem description → INCORRECT
- Real-world knowledge is NOT a valid source.

Generate steps that cover:
- How to identify expected classes from the problem
  description only (not from real-world knowledge)
- How to measure Class Completeness (0-1):
  matched / total expected classes
- How to measure Class Correctness (0-1):
  correct / total candidate classes
- How to verify stereotype correctness
  (abstract, interface, enum)
- How to handle naming variants
  (e.g. OrderMgr vs OrderManager)
- How to handle unmatched candidate classes
  (not supported by problem → INCORRECT)

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (no gold) using {model}...")
    steps = call_llm(prompt, model)

    with open("/cot_steps_classes_no_gold.txt", "w", encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_classes_no_gold.txt")

    return steps


def generate_cot_steps_with_gold(model):
    """
    Generate CoT evaluation steps WITH gold reference.
    Used by S5 and S6.
    Run once — saved to file for inspection and reuse.
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate the CLASSES in a PlantUML candidate diagram against a
gold reference diagram and a problem description.

The evaluator will apply this correctness rule:
- Matched AND stereotype correct → CORRECT
- Not matched but supported by problem → CORRECT
- Not matched AND not supported by problem → INCORRECT
- Real-world knowledge is NOT a valid source.

Generate steps that cover:
- How to match candidate classes to gold classes (exact name first, then semantic match)
- How to measure Class Completeness (0-1):
  matched / total gold classes
- How to measure Class Correctness (0-1):
  correct / total candidate classes
- How to verify stereotype correctness (abstract, interface, enum)
- How to handle naming variants (e.g. OrderMgr vs OrderManager)
- How to handle unmatched candidate classes
  (check problem description before marking incorrect)
- How to handle missing classes
  (present in gold but not in candidate)

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (with gold) using {model}...")
    steps = call_llm(prompt, model)

    with open("/cot_steps_classes_with_gold.txt", "w",
              encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_classes_with_gold.txt")

    return steps

## Reflection

In [ ]:
def needs_reflection(round_num, current_scores,previous_scores=None):
    """
    Decide whether to run a reflection round.

    Rules:
    - Round 1: always runs (previous_scores is None, no delta to compare yet)
    - Round 2: only runs if max score delta between initial scores and round 1 scores > DELTA
    - Never exceeds MAX_REFLECT rounds

    Args:
        round_num:       current round number
        current_scores:  scores from most recent round
        previous_scores: scores from the round before
                         (None for round 1)
    """
    # Never exceed max reflection rounds
    if round_num > MAX_REFLECT:
        return False

    # Round 1 always runs — no previous scores to compare
    if previous_scores is None:
        return True

    # Round 2+ — only run if scores are still changing
    deltas = [
        abs(current_scores[k] - previous_scores[k])
        for k in current_scores
        if current_scores.get(k) is not None
        and previous_scores.get(k) is not None
    ]

    if not deltas:
        return False

    return max(deltas) > DELTA

## **PROMPTS**


In [ ]:
def s1_prompt(problem, candidate):
    """S1 — Simple baseline. No CoT, no gold."""
    return f"""You are evaluating the CLASSES in a PlantUML candidate diagram.

Definitions:
- Completeness (0-1): fraction of expected classes present in the candidate.
- Correctness (0-1): fraction of candidate classes that are accurate.

Correctness Rule:
The ONLY valid reference is the problem description.
- Compare against the problem description.
- Do NOT use real-world domain knowledge as a source.



Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:
"""


In [ ]:
def s2_prompt(problem, candidate, cot_steps):
    """S2 — Auto-CoT, no gold reference."""
    return f"""You are an expert evaluator of UML class diagrams.

    {cot_steps}


Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:
"""


In [ ]:
def s3_reflection_prompt(problem, candidate, cot_steps, previous_scores, round_num):
    """
    S3 — Reflection prompt. No gold reference.
    """


    comp = previous_scores['class_completeness']
    corr = previous_scores['class_correctness']

    prompt = f"""You previously evaluated the CLASSES in this diagram (reflection round {round_num}) and produced:

- Class Completeness: {comp}
- Class Correctness:  {corr}


Please reflect carefully using ONLY the problem
description as your reference:

For Class Completeness:
1. Re-read the problem description carefully.
2. List every class you believe should exist based on the problem.
3. Do NOT use real-world domain knowledge as a source.
4. Check which of those are present in the candidate.
4. Recompute: completeness = present / expected
5. Revise your score if needed.


For Class Correctness:
1. List every class in the candidate diagram.
2. For each one decide if it is appropriate for
   this problem (correct name, correct stereotype).
3. Recompute: correctness = correct / total candidate
4. Revise your score if needed.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:"""

    return prompt


In [ ]:
def s4_prompt(problem, gold, candidate):
    """S4 — Reference only, no CoT."""
    return f"""You are evaluating the CLASSES in a PlantUML candidate diagram against a gold reference.



Definitions:
- Completeness (0-1): fraction of gold classes present in the candidate.
  Formula: matched / total gold classes
- Correctness (0-1): fraction of candidate classes that are accurate compared to gold.
  Formula: correct / total candidate classes


Matching:
A candidate class MATCHES a gold class if its name is identical or semantically equivalent.

Correctness sources (in order):
1. Matched AND stereotype correct → CORRECT
2. Not matched, but supported by problem description
   with appropriate stereotype → CORRECT
3. Not matched AND not supported by problem → INCORRECT
   (counts against correctness)

Real-world knowledge is NOT a valid source.


Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:
"""


In [ ]:
def s5_prompt(problem, gold, candidate, cot_steps):
    """S5 — Reference + Auto-CoT."""
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:
"""


In [ ]:
def s6_reflection_prompt(problem, gold, candidate,cot_steps, previous_scores,round_num):
    """
    S6 — Reflection prompt.
    """
    comp = previous_scores['class_completeness']
    corr = previous_scores['class_correctness']

    return f"""You previously evaluated the CLASSES in this
diagram (reflection round {round_num}) and produced:

- Class Completeness: {comp}
- Class Correctness:  {corr}

Please reflect carefully using the gold reference:

For Class Completeness:
1. List every class in the gold diagram.
2. For each gold class check if it exists in
   the candidate (exact or semantic match).
3. Recompute: completeness = matched / total gold
4. Revise your score if needed.

For Class Correctness:
1. List every class in the candidate diagram.
2. For each candidate class check if it exists in
   the gold AND has the correct stereotype.
3. Recompute: correctness = correct / total candidate
4. Revise your score if needed.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Class Completeness:
- Class Correctness:"""

## **SCORE PARSER**



In [ ]:
import re

def parse_class_scores(response):
    """
    Extract class completeness and correctness from LLM response text.
    """
    scores = {
        'class_completeness': None,
        'class_correctness':  None
    }

    if response is None:
        return scores
    clean_response = response.replace("**", "")

    for line in clean_response.strip().split("\n"):
        line_lower = line.lower()

        match = re.search(r'[\=\:]\s*([\d\.]+)\s*%?', line)
        if not match:
            continue

        try:
            value = float(match.group(1))

            if "%" in line:
                value = value / 100

            value = max(0.0, min(1.0, value))

            if "class completeness" in line_lower:
                scores['class_completeness'] = value
            elif "class correctness" in line_lower:
                scores['class_correctness'] = value

        except ValueError:
            continue

    return scores

In [ ]:
import re

def parse_class_scores(response):
    """
    Extract class completeness and correctness
    from LLM response text.
    """
    scores = {
        'class_completeness': None,
        'class_correctness':  None
    }
    print(response)
    print("###############")
    if response is None:
        return scores

    def extract_value(line):
        """
        Extract a numeric score from a single line.
        Handles decimal (0.8) and fraction (8/10).
        Returns float in [0, 1] or None.
        """
        fraction = re.search(r'(\d+)\s*/\s*(\d+)', line)
        if fraction:
            numerator   = float(fraction.group(1))
            denominator = float(fraction.group(2))
            if denominator > 0:
                return max(0.0, min(1.0,
                           numerator / denominator))
            return None

        decimal = re.search(r':\s*([0-1](?:\.\d+)?)', line)
        if decimal:
            return max(0.0, min(1.0,
                       float(decimal.group(1))))

        return None

    for line in response.strip().split('\n'):
        line_lower = line.lower()

        if 'class completeness' in line_lower:
            scores['class_completeness'] = extract_value(line)

        elif 'class correctness' in line_lower:
            scores['class_correctness'] = extract_value(line)

    return scores

In [ ]:

def  evaluate_classes(scenario, problem, candidate, model,  gold=None, cot_steps_no_gold=None, cot_steps_with_gold=None):
    """
    Evaluate classes for a single candidate
    under a given scenario.

    Returns dict with:
      - class_completeness
      - class_correctness
      - reflected
    """
    reflected = False
    reflection_rounds = 0


    if scenario == 's1':
        prompt   = s1_prompt(problem, candidate)
        response = call_llm(prompt, model)
        scores   = parse_class_scores(response)


    elif scenario == 's2':
        prompt   = s2_prompt(problem, candidate, cot_steps_no_gold)
        response = call_llm(prompt, model)
        scores   = parse_class_scores(response)


    elif scenario == 's3':
        prompt = s2_prompt(problem, candidate, cot_steps_no_gold)
        response   = call_llm(prompt, model)
        scores = parse_class_scores(response)


        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores, previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s3_reflection_prompt(problem, candidate, cot_steps_no_gold,previous_scores, round_num)
            response = call_llm(ref_prompt, model)
            scores   = parse_class_scores(response)


    elif scenario == 's4':
        prompt   = s4_prompt(problem, gold, candidate)
        response = call_llm(prompt, model)
        scores   = parse_class_scores(response)



    elif scenario == 's5':
        prompt   = s5_prompt(problem, gold, candidate,cot_steps_with_gold)
        response = call_llm(prompt, model)
        scores   = parse_class_scores(response)

    elif scenario == 's6':
        prompt = s5_prompt(problem, gold, candidate, cot_steps_with_gold)
        response  = call_llm(prompt, model)
        scores = parse_class_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s6_reflection_prompt( problem, gold, candidate, cot_steps_with_gold, previous_scores, round_num)
            response = call_llm(ref_prompt, model)
            scores   = parse_class_scores(response)
            print('here')
            print(scores)


    else:
        print(f"  Unknown scenario: {scenario}")
        scores = {
            'class_completeness': None,
            'class_correctness':  None
        }

    scores['reflected']         = reflected
    scores['reflection_rounds'] = reflection_rounds
    return scores



In [ ]:
import chardet

def load_gold(filepath):
    with open(filepath, 'rb') as f:
        detected = chardet.detect(f.read())
    encoding = detected['encoding'] or 'windows-1252'

    gold = {}
    with open(filepath, newline='', encoding=encoding) as f:
        reader = csv.DictReader(f)
        for row in reader:
            gold[row['id']] = {
                'problem':  row['problem'],
                'solution': row['solution']
            }
    return gold

In [ ]:
def load_candidates(filepath):
    """
    Load candidates CSV.
    Expected columns:
        id, student_answer,
        score1 (class_completeness),
        score2 (class_correctness),
        score3 (attr_completeness),
        score4 (attr_correctness),
        score5 (rel_completeness),
        score6 (rel_correctness)

    id directly matches gold file id.
    """
    candidates = []
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            candidates.append({
                'id':      row['id'],
                'diagram': row['student_answer'],
                'human_scores': {
                    'class_completeness': row['score1'],
                    'class_correctness':  row['score2'],
                    'attr_completeness':  row['score3'],
                    'attr_correctness':   row['score4'],
                    'rel_completeness':   row['score5'],
                    'rel_correctness':    row['score6']
                }
            })
    return candidates

In [ ]:

def save_results(results, output_filepath):
    """Save LLM scores + human scores to CSV."""
    fieldnames = [
        'candidate_id', 'scenario', 'model',
        'reflected', 'reflection_rounds',
        'human_class_completeness', 'human_class_correctness',
        'llm_class_completeness',   'llm_class_correctness'
    ]
    with open(output_filepath, 'w', newline='',
              encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\n  Results saved to: {output_filepath}")

In [ ]:

def run_class_evaluation(gold_file, candidate_file,scenario, model, output_file):
    """
    Main loop — evaluates CLASS dimension for all
    candidates under a given scenario and model.
    """
    print(f"\n{'='*55}")
    print(f"  CD-EVAL — Class Dimension")
    print(f"  Scenario        : {scenario.upper()}")
    print(f"  Model           : {model}")
    print(f"  Max reflections : {MAX_REFLECT}")
    print(f"  Delta threshold : {DELTA}")
    print(f"{'='*55}")

    # Load data
    print("\nLoading data...")
    candidates = load_candidates(candidate_file)
    gold       = load_gold(gold_file)
    print(f"  Gold scenarios : {len(gold)}")
    print(f"  Candidates     : {len(candidates)}")

    # Generate CoT steps once if needed
    # S2 and S3 share the same no-gold CoT steps
    # S5 and S6 share the same with-gold CoT steps
    cot_steps_no_gold   = None
    cot_steps_with_gold = None

    if scenario in ['s2', 's3']:
        cot_steps_no_gold = generate_cot_steps_no_gold(model)

    elif scenario in ['s5', 's6']:
        cot_steps_with_gold = generate_cot_steps_with_gold(model)

    # Evaluation loop
    results = []
    print(f"\nEvaluating {len(candidates)} candidates...\n")

    for candidate in tqdm(candidates):

        cid          = candidate['id']
        diagram      = candidate['diagram']
        human_scores = candidate['human_scores']

        problem  = gold[cid]['problem']
        solution = gold[cid]['solution']

        # Run evaluation
        llm_scores = evaluate_classes(
            scenario            = scenario,
            problem             = problem,
            candidate           = diagram,
            model               = model,
            gold                = solution,
            cot_steps_no_gold   = cot_steps_no_gold,
            cot_steps_with_gold = cot_steps_with_gold
        )

        # Collect result row
        results.append({
            'candidate_id':    cid,
            'scenario':        scenario,
            'model':           model,
            'reflected':       llm_scores['reflected'],
            'reflection_rounds': llm_scores['reflection_rounds'],
            # human scores
            'human_class_completeness':
                human_scores['class_completeness'],
            'human_class_correctness':
                human_scores['class_correctness'],
            # llm scores
            'llm_class_completeness':
                llm_scores['class_completeness'],
            'llm_class_correctness':
                llm_scores['class_correctness']
        })

        #time.sleep(0.5)

    save_results(results, output_file)

    # Summary
    reflected_count = sum(1 for r in results if r['reflected'])
    print(f"\n  Total evaluated   : {len(results)}")
    print(f"  Reflection triggered: {reflected_count} "
          f"({100*reflected_count//max(len(results),1)}%)")

In [ ]:
gold_file = "GoldDataset.csv"
candidate_file= "CandidatesDataset.csv"
scenario= "s6"
model="claude-opus-4-6"
output_file="ClaudeClassS6.csv"

run_class_evaluation(gold_file, candidate_file,scenario, model, output_file)



  CD-EVAL — Class Dimension
  Scenario        : S6
  Model           : claude-opus-4-6
  Max reflections : 2
  Delta threshold : 0.1

Loading data...
  Gold scenarios : 13
  Candidates     : 13

  Generating CoT steps (with gold) using claude-opus-4-6...
  Saved to cot_steps_classes_with_gold.txt

Evaluating 13 candidates...



  0%|          | 0/13 [00:00<?, ?it/s]

I'll evaluate the candidate diagram against the gold reference diagram by following the detailed steps.

## Phase 1: Preparation

### Gold Classes (G_total = 10):
1. Customer (concrete)
2. InsurancePolicy (concrete)
3. Contract (concrete)
4. Invoice (concrete)
5. Broker (concrete)
6. Claim (concrete)
7. ClaimCase (concrete)
8. Report (concrete)
9. Estimator (concrete)
10. CompensationPayment (concrete)

### Candidate Classes (C_total = 8):
1. Customer (concrete)
2. Broker (concrete)
3. InsurancePolicy (concrete)
4. Contract (concrete)
5. ClaimCase (concrete)
6. Estimator (concrete)
7. Report (concrete)
8. Compensation (concrete)

## Phase 2: Matching

### Exact Matches:
1. Customer ↔ Customer ✓
2. Broker ↔ Broker ✓
3. InsurancePolicy ↔ InsurancePolicy ✓
4. Contract ↔ Contract ✓
5. ClaimCase ↔ ClaimCase ✓
6. Estimator ↔ Estimator ✓
7. Report ↔ Report ✓

### Semantic Matches:
8. Compensation ↔ CompensationPayment — "Compensation" is a reasonable semantic match for "CompensationPayment" (

  8%|▊         | 1/13 [00:19<03:52, 19.39s/it]

I'll carefully re-evaluate the classes by following the detailed instructions.

## Phase 1: Extract All Classes

**Gold classes (G_total = 10):**
1. Customer (concrete)
2. InsurancePolicy (concrete)
3. Contract (concrete)
4. Invoice (concrete)
5. Broker (concrete)
6. Claim (concrete)
7. ClaimCase (concrete)
8. Report (concrete)
9. Estimator (concrete)
10. CompensationPayment (concrete)

**Candidate classes (C_total = 8):**
1. Customer (concrete)
2. Broker (concrete)
3. InsurancePolicy (concrete)
4. Contract (concrete)
5. ClaimCase (concrete)
6. Estimator (concrete)
7. Report (concrete)
8. Compensation (concrete)

## Phase 2: Matching

**Exact matches:**
1. Customer ↔ Customer ✓
2. Broker ↔ Broker ✓
3. InsurancePolicy ↔ InsurancePolicy ✓
4. Contract ↔ Contract ✓
5. ClaimCase ↔ ClaimCase ✓
6. Estimator ↔ Estimator ✓
7. Report ↔ Report ✓

**Semantic matching (remaining):**
- Candidate: Compensation ↔ Gold: CompensationPayment → These are semantically equivalent (both represent the compens

...

I'll carefully re-evaluate the classes by following the detailed instructions.

## Phase 1: Preparation

### Gold classes:
1. City (concrete)
2. Attraction (concrete)
3. Visitor (concrete)
4. GuidedTour (concrete)
5. Discount (concrete)
6. Person (concrete)
7. Guide (concrete)

**G_total = 7**

### Candidate classes:
1. City (concrete)
2. Attraction (concrete)
3. GuidedTour (concrete)
4. Person (concrete)
5. Visitor (concrete)
6. Guide (concrete)
7. Discount (concrete)

**C_total = 7**

## Phase 2: Matching

### Step 2.1: Exact Name Matching
- City ↔ City ✓
- Attraction ↔ Attraction ✓
- GuidedTour ↔ GuidedTour ✓
- Person ↔ Person ✓
- Visitor ↔ Visitor ✓
- Guide ↔ Guide ✓
- Discount ↔ Discount ✓

All 7 gold classes matched. No unmatched classes on either side.

### Step 2.3: Final Matching Results
- **Matched pairs**: 7/7
- **Unmatched candidate classes**: 0
- **Unmatched gold classes**: 0

## Phase 3: Stereotype Verification

| Matched Pair | Gold Stereotype | Candidate Stereotype | Ma

 92%|█████████▏| 12/13 [04:13<00:20, 20.79s/it]

I'll carefully re-evaluate the classes by following the detailed instructions.

## Phase 1: Extract All Classes

**Gold classes (G_total = 18):**
1. Mode (enum)
2. Color (enum)
3. TileO (concrete)
4. Game (concrete)
5. Connection (concrete)
6. Die (concrete)
7. Deck (concrete)
8. Tile (abstract)
9. ActionCard (abstract)
10. RollDieActionCard (concrete)
11. ConnectTilesActionCard (concrete)
12. RemoveConnectionActionCard (concrete)
13. TeleportActionCard (concrete)
14. LoseTurnActionCard (concrete)
15. WinTile (concrete)
16. ActionTile (concrete)
17. NormalTile (concrete)
18. Player (concrete)

**Candidate classes (C_total = 13):**
1. Game (concrete)
2. Board (concrete)
3. Tile (concrete)
4. ActionTile (concrete)
5. ConnectionPiece (concrete)
6. Player (concrete)
7. PlayingPiece (concrete)
8. Die (concrete)
9. Deck (concrete)
10. ActionCard (concrete)
11. Designer (concrete)
12. SparePile (concrete)

## Phase 2: Matching

**Exact matches:**
- Game ↔ Game ✓
- Tile ↔ Tile ✓
- ActionTile ↔

100%|██████████| 13/13 [04:37<00:00, 21.35s/it]

I'll carefully re-evaluate the classes by following the detailed instructions.

## Phase 1: Extract All Classes

**Gold classes:**
1. Employee (abstract)
2. Faculty (concrete)
3. AdministrativeEmployee (concrete)
4. ResearchAssociate (concrete)
5. Institute (concrete)
6. Course (concrete)
7. Lecturer (concrete)
8. Project (concrete)
9. Participation (concrete)

G_total = 9

**Candidate classes:**
1. University (concrete)
2. Faculty (concrete)
3. Institute (concrete)
4. Employee (concrete)
5. AdminStaff (concrete)
6. ResearchAssistant (concrete)
7. Lecturer (concrete)
8. Project (concrete)
9. Course (concrete)
10. Involvement (concrete)

C_total = 10

## Phase 2: Matching

**Exact matches:**
- Faculty ↔ Faculty ✓
- Institute ↔ Institute ✓
- Lecturer ↔ Lecturer ✓
- Project ↔ Project ✓
- Course ↔ Course ✓
- Employee ↔ Employee ✓

**Semantic matches:**
- AdminStaff ↔ AdministrativeEmployee ✓ (synonym)
- ResearchAssistant ↔ ResearchAssociate ✓ (synonym)
- Involvement ↔ Participation ✓ (syno

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="CD-EVAL: Class dimension evaluator"
    )
    parser.add_argument(
        "--gold", required=True,
        help="Path to gold CSV (columns: id, problem, solution)"
    )
    parser.add_argument(
        "--candidates", required=True,
        help="Path to candidates CSV"
    )
    parser.add_argument(
        "--scenario", required=True,
        choices=['s1', 's2', 's3', 's4', 's5', 's6'],
        help="Evaluation scenario"
    )
    parser.add_argument(
        "--model", default="gpt-4",
        choices=list(MODELS.keys()),
        help="LLM to use as evaluator"
    )
    parser.add_argument(
        "--output", required=True,
        help="Path to output CSV file"
    )

    args = parser.parse_args()

    run_class_evaluation(
        gold_file      = args.gold,
        candidate_file = args.candidates,
        scenario       = args.scenario,
        model          = args.model,
        output_file    = args.output
    )